# M11 · Logique, induction et inégalités de concentration — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m11_logique_concentration/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m11_logique_concentration/03_notebook.ipynb`, créée par `python tools/start_chapter.py M11`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M11.1 à M11.10, M11.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M11.17 | Les bornes de Markov et de Tchebychev · `markov_bound, chebyshev_bound` | 🔨 | ★ | 15 |
|  | M11.18 | La borne et le rayon de Hoeffding · `hoeffding_bound, hoeffding_radius` | 🔨 | ★ | 15 |
|  | M11.19 | La queue empirique d'un échantillon · `empirical_tail` | 🔨 | ★ | 10 |
| B | M11.20 | Les vraies queues d'une loi exponentielle et d'une binomiale (`scipy.stats`) contre les bornes | 📦 | ★ | 15 |
|  | M11.21 | Prédire : laquelle des trois bornes est la plus serrée pour n = 100 ? | 🔮 | ★ | 10 |
|  | M11.22 | La fréquence des écarts \|x̄ − p\| ≥ t comparée à la borne de Hoeffding, quand n grandit | 🔬 | ★★ | 20 |
|  | M11.23 | La probabilité qu'un tirage d'une loi Beta dépasse celui d'une autre, par simulation | 🔬 | ★★ | 15 |
| C | M11.24 | Les bornes de Markov, de Tchebychev et de Hoeffding contre la vraie queue, en échelle log | 🎨 | ★ | 15 |
|  | M11.25 | Chasse à l'erreur : Markov pour une variable négative, Hoeffding sans bornes, un rayon sans racine | 🐛 | ★ | 15 |
|  | M11.26 | Un test paramétré qui vérifie l'inégalité de Markov sur dix lois | 🛠️ | ★ | 15 |
|  | M11.27 | Défi : la plus petite taille d'échantillon qui garantit une moyenne à ± 0,05 avec 99 %, vérifiée sur une loi défavorable | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M11")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : tables de vérité, quantificateurs, ensembles, taux de base, Markov, Tchebychev, Hoeffding, bonus d'UCB, ε-greedy.
- Écrire les cinq fonctions de `mymath/concentration.py` et les tester.
- Calculer de vraies queues avec `scipy.stats`, prédire quelle borne gagne, simuler des écarts et des tirages de Thompson, dessiner les bornes, chasser trois erreurs, écrire un test, relever un défi.

**Rappel express.** Valide : aucune ligne « prémisses vraies, conclusion fausse » ; Markov $P(X \ge a) \le \frac{\mathbb{E}[X]}{a}$ ($X \ge 0$) ; Tchebychev $P(\lvert X - \mu \rvert \ge t) \le \frac{\sigma^2}{t^2}$, et $\frac{\sigma^2}{nt^2}$ pour une moyenne ; Hoeffding $P(\lvert \bar{X} - \mu \rvert \ge t) \le 2e^{-2nt^2/(b - a)^2}$ ; rayon $(b - a)\sqrt{\ln(2/\delta)/(2n)}$ ; regret $\sum_k \Delta_k\,\mathbb{E}[N_k(T)]$ ; bonus d'UCB1 $\sqrt{2\ln t / N}$.

## Partie 0 · Vérifier tes exercices papier (✏️ M11.1 à M11.10, 🔗 M11.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `0.62`, arrondi comme le demande la question ;
- `True` / `False` ; une liste `[False, True, True, False]` pour une colonne de table de vérité, `[2, 4]` pour des numéros de lignes, `[0.09, 0.18]` pour les deux bornes d'un intervalle ;
- des lettres entre guillemets, `"AC"`.

Les réponses pas encore remplies affichent ⏳.

**Ex M11.1 — Tables de vérité : modus ponens, modus tollens et l'affirmation du conséquent**

In [ ]:
# Ex M11.1: the values you found on paper
answer_M11_1a = ...  # a) a list of 4 booleans
answer_M11_1b = ...  # b) a list of line numbers, in increasing order
answer_M11_1c = ...  # c) True or False
answer_M11_1d = ...  # d) a list of line numbers, in increasing order
answer_M11_1e = ...  # e) True or False
answer_M11_1f = ...  # f) an integer
answer_M11_1g = ...  # g) True or False
answer_M11_1h = ...  # h) a line number
answer_M11_1i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M11_1a, answer_M11_1b, answer_M11_1c, answer_M11_1d, answer_M11_1e, answer_M11_1f, answer_M11_1g, answer_M11_1h, answer_M11_1i]):
    wbm.check(f"M11.1{letter}", answer)

**Ex M11.2 — Nier des phrases quantifiées sur des modèles et des datasets**

In [ ]:
# Ex M11.2: the values you found on paper
answer_M11_2a = ...  # a) True or False
answer_M11_2b = ...  # b) letters, as text
answer_M11_2c = ...  # c) True or False
answer_M11_2d = ...  # d) True or False
answer_M11_2e = ...  # e) True or False
answer_M11_2f = ...  # f) True or False
answer_M11_2g = ...  # g) a letter, as text
answer_M11_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M11_2a, answer_M11_2b, answer_M11_2c, answer_M11_2d, answer_M11_2e, answer_M11_2f, answer_M11_2g, answer_M11_2h]):
    wbm.check(f"M11.2{letter}", answer)

**Ex M11.3 — Des phrases traduites en inclusions d'ensembles, et un contre-exemple**

In [ ]:
# Ex M11.3: the values you found on paper
answer_M11_3a = ...  # a) a list of 2 integers
answer_M11_3b = ...  # b) True or False
answer_M11_3c = ...  # c) an integer
answer_M11_3d = ...  # d) an integer
answer_M11_3e = ...  # e) a list of integers, in increasing order
answer_M11_3f = ...  # f) True or False
answer_M11_3g = ...  # g) an integer
answer_M11_3h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M11_3a, answer_M11_3b, answer_M11_3c, answer_M11_3d, answer_M11_3e, answer_M11_3f, answer_M11_3g, answer_M11_3h]):
    wbm.check(f"M11.3{letter}", answer)

**Ex M11.4 — Une induction trompeuse par oubli du taux de base, recalculée avec Bayes**

In [ ]:
# Ex M11.4: the values you found on paper
answer_M11_4a = ...  # a) a number (4 decimals)
answer_M11_4b = ...  # b) a number (4 decimals)
answer_M11_4c = ...  # c) a number
answer_M11_4d = ...  # d) a number (4 decimals)
answer_M11_4e = ...  # e) a number (4 decimals)
answer_M11_4f = ...  # f) a number (4 decimals)
answer_M11_4g = ...  # g) a number (1 decimal)

for letter, answer in zip("abcdefg", [answer_M11_4a, answer_M11_4b, answer_M11_4c, answer_M11_4d, answer_M11_4e, answer_M11_4f, answer_M11_4g]):
    wbm.check(f"M11.4{letter}", answer)

**Ex M11.5 — Markov : la queue d'une durée positive**

In [ ]:
# Ex M11.5: the values you found on paper
answer_M11_5a = ...  # a) a number
answer_M11_5b = ...  # b) a number
answer_M11_5c = ...  # c) a number
answer_M11_5d = ...  # d) a number (4 decimals)
answer_M11_5e = ...  # e) a number (1 decimal)
answer_M11_5f = ...  # f) a number
answer_M11_5g = ...  # g) a number
answer_M11_5h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M11_5a, answer_M11_5b, answer_M11_5c, answer_M11_5d, answer_M11_5e, answer_M11_5f, answer_M11_5g, answer_M11_5h]):
    wbm.check(f"M11.5{letter}", answer)

**Ex M11.6 — Tchebychev : combien d'observations pour une moyenne à ± 0,1 ?**

In [ ]:
# Ex M11.6: the values you found on paper
answer_M11_6a = ...  # a) a number
answer_M11_6b = ...  # b) an integer
answer_M11_6c = ...  # c) an integer
answer_M11_6d = ...  # d) a number
answer_M11_6e = ...  # e) a number (4 decimals)
answer_M11_6f = ...  # f) a number
answer_M11_6g = ...  # g) an integer
answer_M11_6h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M11_6a, answer_M11_6b, answer_M11_6c, answer_M11_6d, answer_M11_6e, answer_M11_6f, answer_M11_6g, answer_M11_6h]):
    wbm.check(f"M11.6{letter}", answer)

**Ex M11.7 — Hoeffding : rayon de confiance et taille d'échantillon d'une proportion**

In [ ]:
# Ex M11.7: the values you found on paper
answer_M11_7a = ...  # a) a number (4 decimals)
answer_M11_7b = ...  # b) a list of 2 numbers (4 decimals)
answer_M11_7c = ...  # c) an integer
answer_M11_7d = ...  # d) an integer
answer_M11_7e = ...  # e) a number (4 decimals)
answer_M11_7f = ...  # f) a number
answer_M11_7g = ...  # g) a number (4 decimals)
answer_M11_7h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M11_7a, answer_M11_7b, answer_M11_7c, answer_M11_7d, answer_M11_7e, answer_M11_7f, answer_M11_7g, answer_M11_7h]):
    wbm.check(f"M11.7{letter}", answer)

**Ex M11.8 — Trois bornes pour une même question : Markov, Tchebychev, Hoeffding, et l'approximation normale**

In [ ]:
# Ex M11.8: the values you found on paper
answer_M11_8a = ...  # a) a number (4 decimals)
answer_M11_8b = ...  # b) a number
answer_M11_8c = ...  # c) a number (4 decimals)
answer_M11_8d = ...  # d) a number (4 decimals)
answer_M11_8e = ...  # e) a number (4 decimals)
answer_M11_8f = ...  # f) a letter, as text
answer_M11_8g = ...  # g) a number (4 decimals)
answer_M11_8h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M11_8a, answer_M11_8b, answer_M11_8c, answer_M11_8d, answer_M11_8e, answer_M11_8f, answer_M11_8g, answer_M11_8h]):
    wbm.check(f"M11.8{letter}", answer)

**Ex M11.9 — D'où vient le bonus d'UCB : Hoeffding avec un risque qui décroît avec t**

In [ ]:
# Ex M11.9: the values you found on paper
answer_M11_9a = ...  # a) a number (4 decimals)
answer_M11_9b = ...  # b) a number (4 decimals)
answer_M11_9c = ...  # c) a number (4 decimals)
answer_M11_9d = ...  # d) a number (4 decimals)
answer_M11_9e = ...  # e) an integer
answer_M11_9f = ...  # f) an integer
answer_M11_9g = ...  # g) a number (4 decimals)
answer_M11_9h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M11_9a, answer_M11_9b, answer_M11_9c, answer_M11_9d, answer_M11_9e, answer_M11_9f, answer_M11_9g, answer_M11_9h]):
    wbm.check(f"M11.9{letter}", answer)

**Ex M11.10 — ε-greedy : la loi du bras joué et le regret attendu par pas**

In [ ]:
# Ex M11.10: the values you found on paper
answer_M11_10a = ...  # a) a number
answer_M11_10b = ...  # b) a number
answer_M11_10c = ...  # c) a number
answer_M11_10d = ...  # d) a number (4 decimals)
answer_M11_10e = ...  # e) a number
answer_M11_10f = ...  # f) a number (4 decimals)
answer_M11_10g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M11_10a, answer_M11_10b, answer_M11_10c, answer_M11_10d, answer_M11_10e, answer_M11_10f, answer_M11_10g]):
    wbm.check(f"M11.10{letter}", answer)

**Ex M11.14 — Pont : combien d'exemples de test pour connaître une accuracy à ± 2 points, garanti par Hoeffding ?**

In [ ]:
# Ex M11.14: the values you found on paper
answer_M11_14a = ...  # a) an integer
answer_M11_14b = ...  # b) a number (4 decimals)
answer_M11_14c = ...  # c) a number (4 decimals)
answer_M11_14d = ...  # d) a number (4 decimals)
answer_M11_14e = ...  # e) a number (4 decimals)
answer_M11_14f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M11_14a, answer_M11_14b, answer_M11_14c, answer_M11_14d, answer_M11_14e, answer_M11_14f]):
    wbm.check(f"M11.14{letter}", answer)

## Partie A · Ta librairie : les bornes de Markov, de Tchebychev et de Hoeffding

Fiche §4 à §6. Les cinq fonctions de `mymath/concentration.py`.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate, stats

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M11.17 — Les bornes de Markov et de Tchebychev · `markov_bound, chebyshev_bound` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer les bornes de Markov et de Tchebychev, avec leurs contrôles d'entrée et le plafond à 1.  
**Prérequis :** fiche §4 et §5 · **mymath :** `concentration.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/concentration.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy et `math` sont permis. Une borne est une probabilité : quand la formule dépasse 1, la fonction rend 1. Chaque nombre rendu est un `float` de Python.

Écris, en suivant les docstrings :
- `markov_bound(mean, a)` : $\min\big(1, \frac{\mathbb{E}[X]}{a}\big)$, la borne de Markov sur $P(X \ge a)$ pour une variable positive ; une `ValueError` si `mean < 0` ou `a <= 0` ;
- `chebyshev_bound(variance, t)` : $\min\big(1, \frac{\mathbb{V}(X)}{t^2}\big)$, la borne de Tchebychev sur $P(\lvert X - \mathbb{E}[X] \rvert \ge t)$ ; une `ValueError` si `variance < 0` ou `t <= 0`.

La vérification calcule :
- a) `markov_bound(2.5, 10)` ; b) `markov_bound(4, 3)` ;
- c) `chebyshev_bound(0.36, 1.2)` ; d) `chebyshev_bound(0, 2)` ;
- e) le nom de l'erreur levée par `markov_bound(-1.0, 2.0)` ; f) le nom de l'erreur levée par `chebyshev_bound(1.0, 0.0)` ;
- g) le nom du type de `markov_bound(1, 4)`, en texte.

Dans ta copie : pourquoi b) vaut-il 1, et que rendrait la formule sans le `min` ? Que dit d) d'une variable de variance nulle ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M11")  # reload your saved file

In [ ]:
with checked("M11.17a"):
    wbm.check("M11.17a", mymath.concentration.markov_bound(2.5, 10), computed=True)
with checked("M11.17b"):
    wbm.check("M11.17b", mymath.concentration.markov_bound(4, 3), computed=True)
with checked("M11.17c"):
    wbm.check("M11.17c", mymath.concentration.chebyshev_bound(0.36, 1.2), computed=True)
with checked("M11.17d"):
    wbm.check("M11.17d", mymath.concentration.chebyshev_bound(0, 2), computed=True)
with checked("M11.17e"):
    wbm.check("M11.17e", raised(mymath.concentration.markov_bound, -1.0, 2.0), computed=True)
with checked("M11.17f"):
    wbm.check("M11.17f", raised(mymath.concentration.chebyshev_bound, 1.0, 0.0), computed=True)
with checked("M11.17g"):
    wbm.check("M11.17g", type(mymath.concentration.markov_bound(1, 4)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m11_concentration.py", "markov_bound or chebyshev_bound")

### Ex M11.18 — La borne et le rayon de Hoeffding · `hoeffding_bound, hoeffding_radius` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la borne de Hoeffding et le rayon de confiance qu'on en tire, pour des variables dans un intervalle quelconque.  
**Prérequis :** fiche §6 · **mymath :** `concentration.py` · **Parcours :** C, E

Écris, en suivant les docstrings :
- `hoeffding_bound(n, t, a=0.0, b=1.0)` : $\min\big(1, 2\exp(-\frac{2nt^2}{(b - a)^2})\big)$, la borne bilatérale de Hoeffding sur $P(\lvert \bar{X} - \mu \rvert \ge t)$ ; une `ValueError` si `n < 1`, `t <= 0` ou `a >= b` ;
- `hoeffding_radius(n, delta, a=0.0, b=1.0)` : $(b - a)\sqrt{\frac{\ln(2/\delta)}{2n}}$, l'écart pour lequel la borne vaut $\delta$ ; une `ValueError` si `n < 1`, si `delta` n'est pas dans $]0, 1[$ ou si `a >= b`.

La vérification compare, à 4 décimales :
- a) `hoeffding_bound(250, 0.08)` ; b) `hoeffding_bound(250, 0.08, a=0.0, b=2.0)` ;
- c) `hoeffding_radius(500, 0.1)` ; d) `hoeffding_radius(500, 0.1, a=-1.0, b=1.0)` ;
- e) `hoeffding_bound(500, hoeffding_radius(500, 0.1))` ;
- f) le nom de l'erreur levée par `hoeffding_radius(100, 1.0)` ; g) celui de l'erreur levée par `hoeffding_bound(0, 0.1)`.

Dans ta copie : pourquoi e) redonne-t-il exactement le risque de c) ? Pourquoi b) est-il bien plus grand que a), et d) le double de c) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M11")  # reload your saved file

In [ ]:
with checked("M11.18a"):
    wbm.check("M11.18a", mymath.concentration.hoeffding_bound(250, 0.08), computed=True)
with checked("M11.18b"):
    wbm.check("M11.18b", mymath.concentration.hoeffding_bound(250, 0.08, a=0.0, b=2.0), computed=True)
with checked("M11.18c"):
    wbm.check("M11.18c", mymath.concentration.hoeffding_radius(500, 0.1), computed=True)
with checked("M11.18d"):
    wbm.check("M11.18d", mymath.concentration.hoeffding_radius(500, 0.1, a=-1.0, b=1.0), computed=True)
with checked("M11.18e"):
    wbm.check("M11.18e", mymath.concentration.hoeffding_bound(500, mymath.concentration.hoeffding_radius(500, 0.1)),
              computed=True)
with checked("M11.18f"):
    wbm.check("M11.18f", raised(mymath.concentration.hoeffding_radius, 100, 1.0), computed=True)
with checked("M11.18g"):
    wbm.check("M11.18g", raised(mymath.concentration.hoeffding_bound, 0, 0.1), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m11_concentration.py", "hoeffding")

### Ex M11.19 — La queue empirique d'un échantillon · `empirical_tail` 🔨 ★ ⏱️ 10 min
**Objectif :** estimer une probabilité de queue par la proportion d'un échantillon, et vérifier que cette proportion respecte Markov.  
**Prérequis :** fiche §4 · **mymath :** `concentration.py` · **Parcours :** C

Écris `empirical_tail(samples, threshold)` : la proportion des valeurs de l'échantillon **au moins égales** au seuil, une estimation de $P(X \ge \text{seuil})$ ; une `ValueError` si l'échantillon est vide.

Avec `sample_19 = np.random.default_rng(19).exponential(2.0, size=20_000)` (vingt mille tirages d'une loi exponentielle de moyenne 2), la vérification calcule :
- a) `empirical_tail([3, 8, 1, 8, 5, 2], 5)` ; b) `empirical_tail([3, 8, 1, 8, 5, 2], 8)` (comparé à 4 décimales : ta fonction, elle, n'arrondit pas) ;
- c) `empirical_tail(sample_19, 6.0)` (comparé à 4 décimales) ;
- d) le nom de l'erreur levée par `empirical_tail([], 1.0)` ;
- e) vrai ou faux : pour chaque seuil $a$ de `[0.5, 1, 2, 4, 8, 16]`, `empirical_tail(sample_19, a) <= sample_19.mean() / a`.

Dans ta copie : comparer c) à la valeur exacte $e^{-3} \approx 0{,}0498$ ; d'où vient l'écart ? Pourquoi e) est-il vrai pour **tout** échantillon de valeurs positives, et pas seulement en moyenne ?

In [ ]:
sample_19 = np.random.default_rng(19).exponential(2.0, size=20_000)   # an exponential law of mean 2

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M11")  # reload your saved file

In [ ]:
with checked("M11.19a"):
    wbm.check("M11.19a", mymath.concentration.empirical_tail([3, 8, 1, 8, 5, 2], 5), computed=True)
with checked("M11.19b"):
    wbm.check("M11.19b", mymath.concentration.empirical_tail([3, 8, 1, 8, 5, 2], 8), computed=True)
with checked("M11.19c"):
    wbm.check("M11.19c", mymath.concentration.empirical_tail(sample_19, 6.0), computed=True)
with checked("M11.19d"):
    wbm.check("M11.19d", raised(mymath.concentration.empirical_tail, [], 1.0), computed=True)
with checked("M11.19e"):
    wbm.check("M11.19e", all(mymath.concentration.empirical_tail(sample_19, a) <= sample_19.mean() / a
                             for a in [0.5, 1, 2, 4, 8, 16]), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m11_concentration.py", "empirical_tail")

## Partie B · Les vraies queues, une prédiction et deux simulations

Fiche §4 à §7. `scipy.stats` contre les bornes, la borne qui gagne, la décroissance exponentielle, et la probabilité que Thompson choisisse une option.

### Ex M11.20 — Les vraies queues d'une loi exponentielle et d'une binomiale (`scipy.stats`) contre les bornes 📦 ★ ⏱️ 15 min
**Objectif :** calculer des probabilités de queue exactes avec scipy.stats, et les comparer aux bornes de ta librairie.  
**Prérequis :** fiche §4 à §6 · **Parcours :** C

Avec `scipy.stats` (importé sous le nom `stats`) :
- $T$ suit une loi exponentielle de moyenne 3 (`stats.expon(scale=3)`) : a) $P(T \ge 15)$, avec la méthode `.sf` (*survival function*, $P(T > x)$, égale ici à $P(T \ge x)$) ; b) combien de fois la borne de Markov dépasse cette valeur (2 décimales) ; c) le plus petit seuil $a$ tel que $P(T \ge a) \le 0{,}01$, avec la méthode `.isf` ;
- $S$ suit une loi binomiale $\mathcal{B}(60 ;\ 0{,}25)$ (`stats.binom(60, 0.25)`) et $\bar{X} = \frac{S}{60}$ : d) $P(\lvert \bar{X} - 0{,}25 \rvert \ge 0{,}1)$, c'est-à-dire $P(S \le 9) + P(S \ge 21)$, avec `.cdf` et `.sf` ; e) la borne de Tchebychev de cette probabilité ($\mathbb{V}(\bar{X}) = \frac{0{,}25 \times 0{,}75}{60}$) ; f) la borne de Hoeffding. Calcule e) et f) par la formule de la fiche, sans appeler ta librairie (elles sont celles de `chebyshev_bound` et `hoeffding_bound` : « Run all » doit aller au bout même si ta librairie n'est pas finie).

Les réponses a) et c) à f) sont à 4 décimales. Attention : pour une loi discrète, `.sf(k)` vaut $P(S > k)$, donc $P(S \ge 21)$ s'écrit `.sf(20)`.

Dans ta copie : laquelle des deux bornes de e) et f) est la meilleure ici, et pourquoi Hoeffding perd-elle avec $n = 60$ ?

In [ ]:
answer_M11_20a = ...  # a) P(T >= 15) with stats.expon(scale=3).sf
answer_M11_20b = ...  # b) Markov's bound divided by a)
answer_M11_20c = ...  # c) the smallest a with P(T >= a) <= 0.01, with .isf
answer_M11_20d = ...  # d) P(S <= 9) + P(S >= 21) for S of law B(60, 0.25)
answer_M11_20e = ...  # e) Chebyshev's bound, by the formula of the fiche
answer_M11_20f = ...  # f) Hoeffding's bound, by the formula of the fiche

In [ ]:
for letter, answer in zip("abcdef", [answer_M11_20a, answer_M11_20b, answer_M11_20c, answer_M11_20d, answer_M11_20e,
                                     answer_M11_20f]):
    wbm.check(f"M11.20{letter}", answer)

### Ex M11.21 — Prédire : laquelle des trois bornes est la plus serrée pour n = 100 ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir quelle borne gagne selon la moyenne et l'écart, puis le vérifier.  
**Prérequis :** fiche §4 à §6 · **Parcours :** C, E

$\bar{X}$ est la moyenne de $n = 100$ variables de Bernoulli de paramètre $p$, indépendantes. On cherche une borne de $P(\bar{X} \ge p + t)$ ; trois sont valables :
- **Markov** ($\bar{X} \ge 0$) : $\frac{p}{p + t}$ ;
- **Tchebychev** : $\frac{p(1 - p)}{n t^2}$ (elle borne même $P(\lvert \bar{X} - p \rvert \ge t)$) ;
- **Hoeffding**, unilatérale : $e^{-2nt^2}$.

Pour chaque cas, choisis le numéro de la borne la plus petite (la plus serrée) : (1) Markov ; (2) Tchebychev ; (3) Hoeffding.
- a) $p = 0{,}5$, $t = 0{,}1$ ;
- b) $p = 0{,}02$, $t = 0{,}05$ ;
- c) $p = 0{,}01$, $t = 0{,}5$ ;
- d) $p = 0{,}001$, $t = 0{,}004$.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance le calcul des trois bornes.

In [ ]:
reference_21 = wbm.load_mymath("ref", alias="mymath_ref")     # the experiment uses the reference of concentration.py
CASES_21 = [(0.5, 0.1), (0.02, 0.05), (0.01, 0.5), (0.001, 0.004)]   # (p, t) for the cases a) to d)


def bounds_21(n, p, t):
    """The three bounds on P(X̄ >= p + t) for the mean of n Bernoulli(p): Markov, Chebyshev, one-sided Hoeffding."""
    return [reference_21.concentration.markov_bound(p, p + t),
            reference_21.concentration.chebyshev_bound(p * (1 - p) / n, t),
            min(1.0, math.exp(-2 * n * t ** 2))]

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M11_21a = ...  # a) 1, 2 or 3, written BEFORE running the computation below
prediction_M11_21b = ...  # b) 1, 2 or 3
prediction_M11_21c = ...  # c) 1, 2 or 3
prediction_M11_21d = ...  # d) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcd", [prediction_M11_21a, prediction_M11_21b, prediction_M11_21c,
                                         prediction_M11_21d]):
    wbm.check(f"M11.21{letter}", prediction)

**Le calcul.** Il ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M11_21a, prediction_M11_21b, prediction_M11_21c, prediction_M11_21d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    print(f"{'cas':4} {'Markov':>10} {'Tchebychev':>12} {'Hoeffding':>11}   la plus serrée")
    for letter, (p, t) in zip("abcd", CASES_21):
        bounds = bounds_21(100, p, t)
        best = ["Markov", "Tchebychev", "Hoeffding"][int(np.argmin(bounds))]
        print(f"{letter})   {bounds[0]:10.4g} {bounds[1]:12.4g} {bounds[2]:11.4g}   {best}")

**Après le calcul**, dans ta copie : pour chaque cas, dis ce que chaque borne utilise (la moyenne seule, la variance, ou l'intervalle $[0, 1]$) et pourquoi c'est celle-là qui gagne. Quand $p$ est proche de 0, pourquoi Hoeffding est-elle mauvaise pour un petit écart ? Laquelle choisirais-tu sans rien savoir de $p$ ?

### Ex M11.22 — La fréquence des écarts |x̄ − p| ≥ t comparée à la borne de Hoeffding, quand n grandit 🔬 ★★ ⏱️ 20 min
**Objectif :** simuler la fréquence des grands écarts d'une moyenne, la comparer à la probabilité exacte et à la borne de Hoeffding, et voir la décroissance exponentielle.  
**Prérequis :** fiche §6 · **Parcours :** C

On simule des moyennes $\bar{X}$ de $n$ variables de Bernoulli de paramètre $p = 0{,}25$, et on compte la fréquence des écarts $\lvert \bar{X} - p \rvert \ge t$, avec $t = 0{,}0625 = \frac{1}{16}$ (des nombres exacts en binaire : aucune comparaison de flottants ne tombe à côté).

1. Écris `deviation_frequency(n, p, t, reps, rng)` : elle tire `k = rng.binomial(n, p, size=reps)` (le nombre de succès de `reps` échantillons de taille $n$, en un seul appel), puis rend la proportion des échantillons pour lesquels $\lvert \frac{k}{n} - p \rvert \ge t$, en `float`.
2. La cellule suivante l'appelle pour $n$ dans `[32, 64, 128, 256, 512, 1024]`, dans cet ordre, avec `reps = 4000` et **un seul** générateur `rng = np.random.default_rng(22)` créé avant la boucle ; elle calcule aussi la probabilité exacte (avec `stats.binom`) et la borne de Hoeffding (avec ta fonction).

La vérification calcule :
- a) la fréquence simulée pour $n = 256$ (4 décimales) ;
- b) vrai ou faux : pour chaque $n$, la fréquence simulée est au plus la borne de Hoeffding ;
- c) le plus petit $n$ de la liste pour lequel la borne de Hoeffding est sous 0,05 ;
- d) la probabilité exacte pour $n = 256$ (4 décimales) ;
- e) pour $n = 512$, combien de fois la borne dépasse la probabilité exacte (1 décimale).

Dans ta copie : trace (ou imagine) les trois suites en échelle log : laquelle décroît comme une droite, et pourquoi ? D'où vient l'écart entre a) et d) ? Pourquoi la borne devient-elle de plus en plus lâche en valeur relative (e)) ?

In [ ]:
def deviation_frequency(n, p, t, reps, rng):
    """The share of reps simulated means of n Bernoulli(p) at distance at least t from p (one call to rng.binomial)."""
    raise NotImplementedError("rng.binomial(n, p, size=reps), puis la proportion des |k/n − p| >= t")

In [ ]:
with checked("M11.22"):
    mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M11")
    rng_22, p_22, t_22 = np.random.default_rng(22), 0.25, 0.0625
    rows_22 = []
    for n in [32, 64, 128, 256, 512, 1024]:
        k_exact = np.arange(n + 1)
        exact = float(stats.binom.pmf(k_exact, n, p_22)[np.abs(k_exact / n - p_22) >= t_22].sum())
        rows_22.append((n, deviation_frequency(n, p_22, t_22, 4000, rng_22), exact,
                        mymath.concentration.hoeffding_bound(n, t_22)))
    for n, freq, exact, bound in rows_22:
        print(f"n = {n:5d}  fréquence {freq:.4f}  exacte {exact:.6f}  Hoeffding {bound:.6f}")
    by_n = {n: (freq, exact, bound) for n, freq, exact, bound in rows_22}
    wbm.check("M11.22a", by_n[256][0], computed=True)
    wbm.check("M11.22b", all(freq <= bound for _, freq, _, bound in rows_22), computed=True)
    wbm.check("M11.22c", min(n for n, _, _, bound in rows_22 if bound < 0.05), computed=True)
    wbm.check("M11.22d", by_n[256][1], computed=True)
    wbm.check("M11.22e", by_n[512][2] / by_n[512][1], computed=True)

### Ex M11.23 — La probabilité qu'un tirage d'une loi Beta dépasse celui d'une autre, par simulation 🔬 ★★ ⏱️ 15 min
**Objectif :** estimer par simulation, puis calculer par une intégrale, la probabilité que l'échantillonnage de Thompson choisisse une option plutôt qu'une autre.  
**Prérequis :** fiche §7 ; M04 (loi Beta a posteriori) · **Parcours :** C

Deux options ont, après quelques essais et avec un a priori uniforme, les lois a posteriori $\theta_1 \sim \mathrm{Beta}(a_1, b_1)$ et $\theta_2 \sim \mathrm{Beta}(a_2, b_2)$, indépendantes. L'échantillonnage de Thompson choisit l'option 2 avec la probabilité $P(\theta_2 > \theta_1)$.

1. Écris `prob_draw_greater(a1, b1, a2, b2, size, rng)` : elle tire `x1 = rng.beta(a1, b1, size)`, **puis** `x2 = rng.beta(a2, b2, size)`, et rend la proportion des indices où `x2 > x1`, en `float`.
2. Écris `exact_greater(a1, b1, a2, b2)` : $P(\theta_2 > \theta_1) = \int_0^1 f_2(x)\,F_1(x)\,dx$, où $f_2$ est la densité de $\theta_2$ et $F_1$ la fonction de répartition de $\theta_1$ (`stats.beta(a, b).pdf`, `.cdf`, et `integrate.quad`, qui rend un couple : la valeur, puis une estimation de l'erreur).

La vérification calcule :
- a) `prob_draw_greater(1, 1, 2, 1, 200_000, np.random.default_rng(231))` (4 décimales) ;
- b) la valeur exacte de a), calculée **à la main** (fiche §7 ; 4 décimales) ;
- c) l'option 1 a eu 12 succès et 8 échecs, l'option 2 3 succès et 2 échecs : `prob_draw_greater(13, 9, 4, 3, 200_000, np.random.default_rng(232))` (4 décimales) ;
- d) `exact_greater(13, 9, 4, 3)` (4 décimales) ;
- e) l'option 2 a maintenant 30 succès et 28 échecs (l'option 1 n'a pas changé) : `exact_greater(13, 9, 31, 29)` (4 décimales).

Dans ta copie : les moyennes a posteriori de c) valent 0,59 et 0,57 ; pourquoi Thompson choisit-il quand même l'option 2 presque une fois sur deux ? Que s'est-il passé entre d) et e) ?

In [ ]:
def prob_draw_greater(a1, b1, a2, b2, size, rng):
    """The share of the size pairs of draws (x1 from Beta(a1, b1) first, then x2 from Beta(a2, b2)) with x2 > x1."""
    raise NotImplementedError("x1 = rng.beta(a1, b1, size), puis x2 = rng.beta(a2, b2, size), puis la proportion")


def exact_greater(a1, b1, a2, b2):
    """P(theta2 > theta1) for independent theta1 ~ Beta(a1, b1) and theta2 ~ Beta(a2, b2), by numerical integration."""
    raise NotImplementedError("integrate.quad sur [0, 1] de la densité de theta2 fois la répartition de theta1")


answer_M11_23b = ...  # b) the exact value of a), computed by hand (4 decimals)

In [ ]:
with checked("M11.23a"):
    wbm.check("M11.23a", prob_draw_greater(1, 1, 2, 1, 200_000, np.random.default_rng(231)), computed=True)
wbm.check("M11.23b", answer_M11_23b)
with checked("M11.23c"):
    wbm.check("M11.23c", prob_draw_greater(13, 9, 4, 3, 200_000, np.random.default_rng(232)), computed=True)
with checked("M11.23d"):
    wbm.check("M11.23d", exact_greater(13, 9, 4, 3), computed=True)
with checked("M11.23e"):
    wbm.check("M11.23e", exact_greater(13, 9, 31, 29), computed=True)

## Partie C · Une figure, des erreurs classiques, un test et un défi

Fiche §4 à §6. Les trois bornes sur une figure, trois erreurs à trouver, un test de propriété, puis la taille d'échantillon exacte d'une loi défavorable.

### Ex M11.24 — Les bornes de Markov, de Tchebychev et de Hoeffding contre la vraie queue, en échelle log 🎨 ★ ⏱️ 15 min
**Objectif :** voir sur une figure à quel point chaque borne est large, et comment elles se croisent quand l'écart grandit.  
**Prérequis :** fiche §4 à §6 ; Ex M11.21 · **Parcours :** C

Écris `draw_24(ax, n, p, ts)`, qui trace dans l'axe `ax`, en fonction de $t$ (le tableau `ts`), pour la moyenne $\bar{X}$ de $n$ variables de Bernoulli de paramètre $p$ :
- la **vraie queue** $P(\bar{X} \ge p + t) = P(S \ge n(p + t))$, avec $S \sim \mathcal{B}(n, p)$ : `stats.binom.sf(k - 1, n, p)` où `k = math.ceil(n * (p + t) - 1e-9)` est le plus petit entier au moins égal à $n(p + t)$ ;
- la borne de **Markov** $\frac{p}{p + t}$, la borne de **Tchebychev** $\min\big(1, \frac{p(1 - p)}{n t^2}\big)$ et la borne de **Hoeffding** unilatérale $e^{-2nt^2}$ ;

chacune par `ax.plot(ts, valeurs, label=...)`, avec l'axe des probabilités en échelle log (`ax.set_yscale("log")`) et une légende. La cellule de la figure s'en sert pour $n = 50$ et $p = 0{,}3$ ; une cellule de vérification l'essaie sur d'autres valeurs, dans des axes neufs.

Dans ta copie : pour quels écarts $t$ chaque borne est-elle la meilleure ? Laquelle a la même forme que la vraie queue en échelle log, et pourquoi ?

In [ ]:
def draw_24(ax, n, p, ts):
    """In ax, against t (the array ts): the true tail P(mean >= p + t) for the mean of n Bernoulli(p), and the bounds
    of Markov, Chebyshev and (one-sided) Hoeffding; log scale for the probabilities, and a legend."""
    raise NotImplementedError("quatre ax.plot(ts, ...), ax.set_yscale(\"log\"), ax.legend()")

In [ ]:
with checked("M11.24 figure"):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    try:
        created = drawn(draw_24, ax, 50, 0.3, np.linspace(0.011, 0.491, 49))
    except BaseException:
        plt.close(fig)
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M11.24 : draw_24 ouvre sa propre figure : dessine dans l'axe reçu.")
    else:
        ax.set_xlabel("écart t")
        plt.show()

**Vérifications automatiques** (sur d'autres valeurs, dans des axes neufs).

In [ ]:
with checked("M11.24 vérifications"):
    problems = []
    n_t, p_t = 80, 0.35
    ts_t = np.linspace(0.021, 0.401, 25)               # n (p + t) is never an integer here
    expected = {
        "la vraie queue P(X̄ >= p + t) (stats.binom)": [float(stats.binom.sf(math.ceil(n_t * (p_t + t) - 1e-9) - 1,
                                                                            n_t, p_t)) for t in ts_t],
        "la borne de Markov p / (p + t)": [min(1.0, p_t / (p_t + t)) for t in ts_t],
        "la borne de Tchebychev p(1 − p) / (n t²), plafonnée à 1": [min(1.0, p_t * (1 - p_t) / (n_t * t ** 2))
                                                                    for t in ts_t],
        "la borne de Hoeffding exp(−2 n t²)": [math.exp(-2 * n_t * t ** 2) for t in ts_t],
    }
    fig_t, ax_t = plt.subplots()
    try:
        created = drawn(draw_24, ax_t, n_t, p_t, ts_t)
    finally:
        plt.close(fig_t)
    if created:
        problems.append("draw_24 ouvre sa propre figure : dessine dans l'axe reçu")
    else:
        curves = [np.asarray(line.get_ydata(), dtype=float) for line in ax_t.get_lines()]
        for what, values in expected.items():
            if not any(len(c) == len(values) and np.allclose(c, values, rtol=1e-6, atol=1e-12) for c in curves):
                problems.append(f"il manque {what}, tracée en fonction de t (une ligne ax.plot sur les ts reçus)")
        if ax_t.get_yscale() != "log":
            problems.append("l'axe des probabilités doit être en échelle log : ax.set_yscale(\"log\")")
        if not ax_t.get_legend():
            problems.append("ajoute une légende (ax.legend()) : quatre courbes sans nom ne se lisent pas")
    if problems:
        print("❌ Ex M11.24 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M11.24 : la vraie queue et les trois bornes sont à leur place, en échelle log, avec une légende.")

### Ex M11.25 — Chasse à l'erreur : Markov pour une variable négative, Hoeffding sans bornes, un rayon sans racine 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code trois erreurs classiques sur les inégalités de concentration, et calculer ce qu'elles coûtent.  
**Prérequis :** fiche §4 à §6 · **Parcours :** C

Un collègue a écrit le code ci-dessous pour les variations quotidiennes $x_i$ d'un indicateur, dont on sait qu'elles sont toujours comprises entre $-5$ et $5$ (elles peuvent être négatives). Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
def report_25(x, threshold, t, delta):
    '''Bounds for n daily changes x, each known to lie in [-5, 5], independent.'''
    n = len(x)
    mean = float(np.mean(x))                                   # (A) the sample mean
    markov = min(1.0, mean / threshold)                        # (B) a bound on P(X >= threshold) for one change X
    variance = float(np.var(x))                                # (C) the sample variance (ddof=0), estimates V(X)
    chebyshev = min(1.0, variance / (n * t ** 2))              # (D) Chebyshev's formula, V(X) replaced by (C)
    hoeffding = min(1.0, 2 * math.exp(-2 * n * t ** 2))        # (E) a bound on P(|mean - mu| >= t)
    radius = 10 * math.log(2 / delta) / (2 * n)                # (F) the Hoeffding radius at risk delta
    return markov, chebyshev, hoeffding, radius
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) la vraie borne de Hoeffding sur $P(\lvert \bar{X} - \mu \rvert \ge 0{,}5)$ pour $n = 400$ variations (4 décimales) ;
- c) le vrai rayon de Hoeffding pour $n = 400$ et $\delta = 0{,}05$ (4 décimales) ;
- d) ce que rend la ligne (F) **telle quelle** pour $n = 400$ et $\delta = 0{,}05$ (4 décimales).

Dans ta copie : corrige les trois lignes. Que vaudrait `markov` pour des variations de moyenne $-0{,}2$, et pourquoi ce nombre n'est-il une borne de rien ? Le rayon faux de d) est-il trop grand ou trop petit, et qu'est-ce que cela fait croire ?

In [ ]:
answer_M11_25a = ...  # a) letters, as text
answer_M11_25b = ...  # b) a number (4 decimals)
answer_M11_25c = ...  # c) a number (4 decimals)
answer_M11_25d = ...  # d) a number (4 decimals)

In [ ]:
wbm.check("M11.25a", answer_M11_25a)
wbm.check("M11.25b", answer_M11_25b)
wbm.check("M11.25c", answer_M11_25c)
wbm.check("M11.25d", answer_M11_25d)

### Ex M11.26 — Un test paramétré qui vérifie l'inégalité de Markov sur dix lois 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test de propriété qui vérifie une borne sur plusieurs lois, et choisir des lois qui attrapent les fausses bornes.  
**Prérequis :** fiche §4 ; Ex M11.17 · **Parcours :** C

Une borne se teste par sa **propriété** : elle doit rester au-dessus de la vraie probabilité, pour toute loi qui vérifie ses hypothèses. Écris `test_markov_bound_on_ten_laws()`, **sans argument** (« paramétré » veut dire ici : une boucle sur une liste de lois **dans** la fonction, pas `@pytest.mark.parametrize`), qui vérifie, pour **dix lois positives** de `scipy.stats` (continues ou discrètes, gelées avec leurs paramètres, comme `stats.expon(scale=2)` ou `stats.bernoulli(0.3)`) et pour plusieurs seuils $a > 0$ chacune, que `markov_bound(law.mean(), a) >= P(X >= a)`, avec une petite tolérance (`1e-12`) et un message d'`assert` qui dit la loi et le seuil.

Attention : pour une loi discrète, `law.sf(a)` vaut $P(X > a)$ ; pour un seuil entier, $P(X \ge a)$ s'écrit `law.sf(a - 1)`, ou bien choisis des seuils non entiers (pour une loi à valeurs entières, $P(X \ge 2{,}5) = P(X > 2{,}5)$).

pytest exécute ta fonction de test **seule**, sans les autres cellules : seuls `math`, `np`, `pytest`, `stats` (`from scipy import stats`) et `markov_bound` lui sont fournis. La vérification la lance trois fois : sur la vraie borne de Markov, où elle doit **passer** ; puis sur `markov_bound_half` et `markov_bound_exp` (fournies), deux fausses bornes, où elle doit **échouer**.

Dans ta copie : quelle loi de ta liste attrape `markov_bound_half` ? Pourquoi une liste de lois « ordinaires » (exponentielle, uniforme, Poisson) peut-elle laisser passer une borne deux fois trop petite ?

In [ ]:
markov_bound = wbm.load_mymath("ref", alias="mymath_ref").concentration.markov_bound   # the true bound, to run your test here


def markov_bound_half(mean, a):
    """A wrong Markov bound: two times too small."""
    return min(1.0, mean / (2 * a))


def markov_bound_exp(mean, a):
    """A wrong Markov bound: the tail of an exponential law of the same mean, exp(-a / mean)."""
    return math.exp(-a / mean) if mean > 0 else 0.0

In [ ]:
def test_markov_bound_on_ten_laws():
    """markov_bound(law.mean(), a) >= P(X >= a) for ten non-negative laws of scipy.stats and several thresholds."""
    raise NotImplementedError("dix lois positives, plusieurs seuils, un assert avec un message par cas")

In [ ]:
import inspect

with checked("M11.26 vérifications"):
    if inspect.signature(test_markov_bound_on_ten_laws).parameters:
        raise TypeError("ton test doit s'appeler sans argument : boucle sur tes lois et tes seuils dans la fonction "
                        "(pas de @pytest.mark.parametrize)")
    test_markov_bound_on_ten_laws()                     # your test runs here first: ⏳ while it is not written
    reference_26 = wbm.load_mymath("ref", alias="mymath_ref")
    results = {
        "la vraie borne de Markov": (reference_26.concentration.markov_bound, True),
        "une borne deux fois trop petite": (markov_bound_half, False),
        "la queue d'une loi exponentielle prise pour la borne": (markov_bound_exp, False),
    }
    problems = []
    for what, (function, should_pass) in results.items():
        run = wbm.run_pytest(test_markov_bound_on_ten_laws, subject=function, name="markov_bound",
                             preamble="from scipy import stats", quiet=True)
        if "NameError" in run.output:
            problems.append("ton test utilise un nom défini hors de lui (NameError) : pytest l'exécute seul ; seuls "
                            "math, np, pytest, stats et markov_bound lui sont fournis")
            break
        if should_pass and not run.ok:
            problems.append(f"ton test échoue sur {what} : il refuse une fonction juste (lis print(run.output))")
        if not should_pass and run.failed == 0:
            problems.append(f"ton test passe sur {what} : il ne la voit pas ; ajoute une loi où Markov est presque "
                            f"atteinte (une loi à deux valeurs, comme une Bernoulli) ou une loi à queue lourde ; pour une loi "
                            f"discrète à un seuil entier, P(X ≥ a) s'écrit law.sf(a - 1), pas law.sf(a)")
    if problems:
        print("❌ Ex M11.26 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M11.26 : ton test accepte la vraie borne et attrape les deux fausses.")

### Ex M11.27 — Défi : la plus petite taille d'échantillon qui garantit une moyenne à ± 0,05 avec 99 %, vérifiée sur une loi défavorable 🏆 ★★★ ⏱️ 25 min
**Objectif :** calculer exactement la taille d'échantillon nécessaire pour une loi défavorable, et la comparer à la garantie de Hoeffding.  
**Prérequis :** fiche §5 et §6 ; Ex M11.22 · **Parcours :** C

Pour des variables à valeurs dans $[0, 1]$, Hoeffding garantit $P(\lvert \bar{X} - \mu \rvert \ge 0{,}05) \le 0{,}01$ dès que $n \ge \frac{\ln 200}{2 \times 0{,}05^2} \approx 1\,059{,}7$, donc pour $n \ge 1\,060$, **quelle que soit** la loi. Est-ce beaucoup trop ? Regardons une loi défavorable : la loi de Bernoulli de paramètre $\frac{1}{2}$, celle dont la variance est la plus grande.

**Le défi.** Écris `smallest_n_27()`, qui rend le plus petit entier $n_0$ tel que, pour **tout** $n$ de $n_0$ à 3 000, la probabilité **exacte** $P\big(\lvert \bar{X} - \frac{1}{2} \rvert \ge 0{,}05\big)$ soit au plus 0,01, où $\bar{X}$ est la moyenne de $n$ variables de Bernoulli$(\frac{1}{2})$ indépendantes. Objectif : la valeur exacte. Deux pièges t'attendent : la probabilité n'est pas une fonction décroissante de $n$ (le nombre de valeurs possibles de $\bar{X}$ change avec $n$), et l'égalité $\lvert \frac{k}{n} - \frac{1}{2} \rvert = 0{,}05$ tombe souvent pile, là où les flottants arrondissent.

Ensuite, dans ta copie : compare ton $n_0$ à la garantie de Hoeffding (1 060) et à l'approximation normale ($n \approx (2{,}576 \times 0{,}5 / 0{,}05)^2 \approx 664$). Pourquoi Hoeffding demande-t-il plus ? Pour ton $n_0$, vérifie que la loi de Bernoulli de paramètre $p$ donne une probabilité plus petite pour $p$ = 0,3 ou 0,1 : pourquoi parle-t-on de loi « défavorable » ?

In [ ]:
def smallest_n_27():
    """The smallest n0 such that P(|mean - 1/2| >= 0.05) <= 0.01 for every n from n0 to 3000, for the mean of n
    independent Bernoulli(1/2) (exact binomial probabilities)."""
    raise NotImplementedError("pour chaque n, la probabilité exacte (stats.binom) ; puis le plus petit n0 qui marche "
                              "jusqu'à 3 000")

In [ ]:
with checked("M11.27"):
    n_27 = smallest_n_27()                   # ⏳ while smallest_n_27 is not written
    wbm.check("M11.27", int(n_27), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu décider par une table de vérité si un argument est valide, et nier une phrase quantifiée ?
2. Sais-tu traduire une phrase en inclusion d'ensembles, et trouver un contre-exemple ?
3. Sais-tu repérer et chiffrer un piège d'induction (taux de base, échantillon biaisé) ?
4. Sais-tu démontrer Markov et Tchebychev, et en tirer une taille d'échantillon ?
5. Sais-tu calculer un rayon de Hoeffding, et dire ce que l'inégalité suppose ?
6. Sais-tu expliquer d'où vient le bonus d'UCB, et pourquoi ε-greedy à ε fixé a un regret linéaire ?

**La suite** : le chapitre 11 de workbookIA (apprentissage et raisonnement, puis un agent qui apprend par essais et récompenses), puis CP2, l'examen blanc de la partie II.